# Clase 064 — Class imbalance: SMOTE, ADASYN, class_weight

Binary sintético con ratio 1:99. Comparamos baseline vs class_weight vs SMOTE vs ADASYN, en métricas relevantes (PR-AUC, recall, no accuracy).

Instalar: `pip install imbalanced-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                              average_precision_score, precision_recall_curve)

try:
    from imblearn.over_sampling import SMOTE, ADASYN
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_OK = True
except ImportError:
    print('instalar: pip install imbalanced-learn')
    IMBLEARN_OK = False

np.random.seed(42)

## 1. Dataset 1:99

In [ ]:
X, y = make_classification(n_samples=10_000, n_features=20, n_informative=8,
                            n_redundant=4, weights=[0.99, 0.01], flip_y=0.02,
                            random_state=42)
print('pos rate:', y.mean().round(4), '— ratio 1:', round((y == 0).sum() / max((y == 1).sum(), 1), 1))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 2. Baseline (sin tratamiento) — accuracy engañosa

In [ ]:
base = LogisticRegression(max_iter=500, random_state=42)
base.fit(X_train, y_train)
preds = base.predict(X_test)
proba = base.predict_proba(X_test)[:, 1]
print(f'accuracy: {accuracy_score(y_test, preds):.4f}   (¡trivial "todo-negativo" también da {1 - y_test.mean():.4f}!)')
print(f'recall minoritario: {recall_score(y_test, preds):.4f}')
print(f'precision minoritario: {precision_score(y_test, preds, zero_division=0):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, proba):.4f}')

## 3. Tres tratamientos

(a) `class_weight='balanced'`, (b) SMOTE, (c) ADASYN — todos dentro de un imblearn Pipeline para que el resampling solo afecte al fold de train durante CV.

In [ ]:
def evaluate(name, model_or_pipeline, X_train, X_test, y_train, y_test):
    model_or_pipeline.fit(X_train, y_train)
    proba = model_or_pipeline.predict_proba(X_test)[:, 1]
    preds = (proba > 0.5).astype(int)
    return {
        'estrategia': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall':   recall_score(y_test, preds),
        'PR_AUC':   average_precision_score(y_test, proba),
    }, proba

results = []
probas = {}
r, p = evaluate('baseline', LogisticRegression(max_iter=500, random_state=42),
                 X_train, X_test, y_train, y_test)
results.append(r); probas['baseline'] = p

r, p = evaluate('class_weight=balanced',
                 LogisticRegression(max_iter=500, class_weight='balanced', random_state=42),
                 X_train, X_test, y_train, y_test)
results.append(r); probas['class_weight'] = p

if IMBLEARN_OK:
    pipe_smote = ImbPipeline([
        ('smote', SMOTE(k_neighbors=5, random_state=42)),
        ('clf',   LogisticRegression(max_iter=500, random_state=42)),
    ])
    r, p = evaluate('SMOTE + LogReg', pipe_smote, X_train, X_test, y_train, y_test)
    results.append(r); probas['SMOTE'] = p

    pipe_adasyn = ImbPipeline([
        ('adasyn', ADASYN(n_neighbors=5, random_state=42)),
        ('clf',    LogisticRegression(max_iter=500, random_state=42)),
    ])
    r, p = evaluate('ADASYN + LogReg', pipe_adasyn, X_train, X_test, y_train, y_test)
    results.append(r); probas['ADASYN'] = p

print(pd.DataFrame(results).round(4).to_string(index=False))

## 4. Curvas Precision-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, prob in probas.items():
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec, prec, label=f'{name} (AP={ap:.3f})')
ax.axhline(y_test.mean(), color='gray', ls='--', label=f'baseline (pos rate {y_test.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas PR — imbalance 1:99')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Recall@K (top-K por probabilidad)

Métrica de negocio: ¿qué fracción de positivos recupero si reviso los top-K predichos?

In [ ]:
def recall_at_k(y_true, proba, k):
    idx_top = np.argsort(-proba)[:k]
    return y_true[idx_top].sum() / max(y_true.sum(), 1)

K = 200
rows = [{'estrategia': n, f'recall@{K}': recall_at_k(y_test, p, K)} for n, p in probas.items()]
print(pd.DataFrame(rows).round(4).to_string(index=False))

## 6. CV correcto con imblearn Pipeline

Importante: si SMOTE se aplica antes del CV, hay leakage. Dentro del Pipeline imblearn, se aplica solo al fold de entrenamiento.

In [ ]:
if IMBLEARN_OK:
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    pipe_cv = ImbPipeline([
        ('smote', SMOTE(k_neighbors=5, random_state=42)),
        ('clf',   LogisticRegression(max_iter=500, random_state=42)),
    ])
    scores = cross_val_score(pipe_cv, X_train, y_train, cv=cv,
                              scoring='average_precision', n_jobs=1)
    print(f'CV PR-AUC (SMOTE solo en train fold): {scores.mean():.4f} ± {scores.std():.4f}')

## Ejercicios

1. Agregá `SMOTETomek` (combina over + under). ¿Mejora PR-AUC?
2. Tuneá el threshold con la curva PR para maximizar F2 (favor recall).
3. Probá con `SMOTENC` si agregás features categóricas.

## Conclusiones

- En 1:99, la accuracy es inútil — reportar PR-AUC, recall, MCC.
- `class_weight='balanced'` es la opción más barata; SMOTE/ADASYN ayudan en casos extremos.
- Aplicar SMOTE antes del split es leakage clásico; usar `imblearn.Pipeline`.